In [8]:
!pip install -q optuna

In [9]:
import os
import requests
import optuna
import pandas as pd
import matplotlib.pyplot as plt
from optuna.importance import get_param_importances
from collections import defaultdict
import re

# ---------------------------
# Configuration
# ---------------------------
OWNER = "econdatatech"
REPO = "Data588"
PATH = "experiments"

DOWNLOAD_DIR = "downloaded_optuna_dbs_new"
RESULTS_DIR = "importance_results_new"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------
# GitHub API helper
# ---------------------------
def github_contents(path):
    url = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{path}"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

# ---------------------------
# Get experiment folders
# ---------------------------
folders = github_contents(PATH)

experiment_folders = [
    item for item in folders
    if item["type"] == "dir"
]


print(f"Found {len(experiment_folders)} experiment folders.")


Found 1 experiment folders.


In [10]:
EXCLUDED_STUDIES = {
    "OVA_ProstateXGBoostNoneRandom0588508",
    "OVA_OmentumXGBoostNoneRandom0588508",
    "OVA_OmentumXGBoostNoneRFsmote083667",
    "OVA_OmentumXGBoostNoneRFsmote1896865",
    "OVA_OmentumXGBoostNoneRFsmote2244098",
}

# ---------------------------
# Main loop over folders
# ---------------------------
for folder in experiment_folders:

    folder_name = folder["name"]

    print("\n" + "=" * 70)
    print(f"Processing folder: {folder_name}")

    try:
        contents = github_contents(f"{PATH}/{folder_name}")

        db_files = [
            f for f in contents
            if f["type"] == "file" and f["name"].endswith(".db")
        ]

        if not db_files:
            print("No DB found.")
            continue

        db_info = db_files[0]
        db_url = db_info["download_url"]
        db_name = db_info["name"]

        local_db = os.path.join(DOWNLOAD_DIR, folder_name+db_name)

        # ---------------------------
        # Download DB
        # ---------------------------
        print(f"Downloading {db_name}")

        r = requests.get(db_url)
        r.raise_for_status()

        with open(local_db, "wb") as f:
            f.write(r.content)

        storage = f"sqlite:///{local_db}"

        # ---------------------------
        # Load studies
        # ---------------------------
        studies = optuna.study.get_all_study_summaries(storage)

        if not studies:
            print("No studies in DB.")
            continue

        print(f"Found {len(studies)} studies.")

        # ---------------------------
        # Group studies by base name
        # ---------------------------
        study_groups = defaultdict(list)

        for s in studies:
            name = s.study_name
            # exclude specific bad runs
            if name in EXCLUDED_STUDIES:
              continue

            # remove trailing 7-digit run id
            base_name = re.sub(r"\d+$", "", name)
            study_groups[base_name].append(name)

        print(f"Grouped into {len(study_groups)} study families.")

        # ---------------------------
        # Process each group
        # ---------------------------
        for base_name, study_names in study_groups.items():

            print(f"\nBase study: {base_name}")
            print(f"Replicates: {len(study_names)}")

            importance_frames = []

            for study_name in study_names:

                try:
                    study = optuna.load_study(
                        study_name=study_name,
                        storage=storage
                    )

                    importance = get_param_importances(study)

                    if not importance:
                        continue

                    df = pd.DataFrame({
                        "parameter": list(importance.keys()),
                        "importance": list(importance.values())
                    })

                    importance_frames.append(df)

                except Exception as e:
                    print(f"  Failed study {study_name}: {e}")

            if not importance_frames:
                print("  No valid studies in group.")
                continue

            # ---------------------------
            # Average importance
            # ---------------------------
            combined = pd.concat(importance_frames, ignore_index=True)

            averaged = (
                combined
                .groupby("parameter", as_index=False)["importance"]
                .mean()
                .sort_values("importance", ascending=False)
            )

            safe_base = base_name.replace("/", "_").replace(" ", "_")

            out_file = os.path.join(
                RESULTS_DIR,
                base_name+"_importance.csv"
            )

            averaged.to_csv(out_file, index=False)

            print(f"  Saved: {out_file}")

    except Exception as e:
        print(f"Folder failed: {e}")

print("\nDone.")


Processing folder: HPClinearOSXGBoost
Found 110 studies.
Grouped into 21 study families.

Base study: OVA_EndometriumXGBoostNoneRandom
Replicates: 5
  Saved: importance_results_new/OVA_EndometriumXGBoostNoneRandom_importance.csv

Base study: OVA_EndometriumXGBoostNoneSMOTE
Replicates: 5
  Saved: importance_results_new/OVA_EndometriumXGBoostNoneSMOTE_importance.csv

Base study: OVA_EndometriumXGBoostNoneADASYN
Replicates: 5
  Saved: importance_results_new/OVA_EndometriumXGBoostNoneADASYN_importance.csv

Base study: OVA_EndometriumXGBoostNoneBorderline
Replicates: 5
  Saved: importance_results_new/OVA_EndometriumXGBoostNoneBorderline_importance.csv

Base study: OVA_EndometriumXGBoostNoneSVM
Replicates: 5
  Saved: importance_results_new/OVA_EndometriumXGBoostNoneSVM_importance.csv

Base study: OVA_ProstateXGBoostNoneRandom
Replicates: 5
  Saved: importance_results_new/OVA_ProstateXGBoostNoneRandom_importance.csv

Base study: OVA_ProstateXGBoostNoneSMOTE
Replicates: 5
  Saved: importance_

In [11]:
!zip -r /content/file_new.zip /content/importance_results_new
from google.colab import files
files.download("/content/file_new.zip")

  adding: content/importance_results_new/ (stored 0%)
  adding: content/importance_results_new/OVA_OmentumLogistic RegressionNoneBorderline_importance.csv (deflated 16%)
  adding: content/importance_results_new/OVA_EndometriumXGBoostNoneBorderline_importance.csv (deflated 26%)
  adding: content/importance_results_new/OVA_ProstateRandom ForestNoneSMOTE_importance.csv (deflated 14%)
  adding: content/importance_results_new/OVA_OmentumSVM (Linear)ica_importance.csv (deflated 1%)
  adding: content/importance_results_new/OVA_ProstateSVM (Linear)NoneBorderline_importance.csv (deflated 14%)
  adding: content/importance_results_new/OVA_EndometriumXGBoostpca_importance.csv (deflated 22%)
  adding: content/importance_results_new/OVA_OmentumSVM (Linear)None_importance.csv (stored 0%)
  adding: content/importance_results_new/OVA_OmentumSVM (Linear)pca_importance.csv (deflated 1%)
  adding: content/importance_results_new/OVA_Prostatek-NNNoneRF_importance.csv (deflated 22%)
  adding: content/importa

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>